# Train a NeRF from scratch — `tiny_nerf`

**Track B · 3D & Neural Rendering** · maps to lessons B5 (NeRF) and B9 (reproduction).

This is a *complete, real* NeRF: a positional-encoded MLP maps `(x,y,z) → (rgb, σ)`, rendered by volume integration along camera rays and trained by a photometric loss against 106 posed photos. Pure PyTorch — no external rendering library.

> **Runtime → Change runtime type → T4 GPU** (it runs on CPU too, just slowly). On a T4, ~3k steps (a few minutes) reaches a sharp render.

In [ ]:
import os, time, numpy as np, torch, torch.nn as nn, matplotlib.pyplot as plt, urllib.request
device = "cuda" if torch.cuda.is_available() else "cpu"
STEPS  = int(os.environ.get("STEPS", 3000))   # raise for a sharper result
print("device:", device, "| steps:", STEPS)

## 1 · Data — 106 posed photos of a Lego scene (~12 MB)
Each image comes with its camera-to-world pose; one view is held out for testing.

In [ ]:
if not os.path.exists("tiny_nerf_data.npz"):
    urllib.request.urlretrieve("https://bmild.github.io/nerf/tiny_nerf_data.npz", "tiny_nerf_data.npz")
d = np.load("tiny_nerf_data.npz")
images = torch.tensor(d["images"], dtype=torch.float32)
poses  = torch.tensor(d["poses"],  dtype=torch.float32)
focal  = float(d["focal"]); H, W = images.shape[1], images.shape[2]
testimg, testpose = images[101].to(device), poses[101].to(device)
images, poses = images[:100].to(device), poses[:100].to(device)
print("images", tuple(images.shape), "| focal", round(focal, 1))
plt.imshow(images[0].cpu()); plt.title("one training view"); plt.axis("off"); plt.show()

## 2 · Model — positional encoding + a small MLP
The encoding lifts each 3D point into sines/cosines at several frequencies so the MLP can represent sharp detail.

In [ ]:
def posenc(x, L=6):
    out = [x]
    for i in range(L):
        for fn in (torch.sin, torch.cos):
            out.append(fn(2.0 ** i * x))
    return torch.cat(out, -1)

class NeRF(nn.Module):
    def __init__(self, L=6, Wd=128):
        super().__init__()
        din = 3 * (1 + 2 * L)
        self.L = L
        self.net = nn.Sequential(
            nn.Linear(din, Wd), nn.ReLU(),
            nn.Linear(Wd, Wd), nn.ReLU(),
            nn.Linear(Wd, Wd), nn.ReLU(),
            nn.Linear(Wd, 4))
    def forward(self, x):
        return self.net(posenc(x, self.L))

def get_rays(H, W, focal, c2w):
    i, j = torch.meshgrid(torch.arange(W, device=device, dtype=torch.float32),
                          torch.arange(H, device=device, dtype=torch.float32), indexing="xy")
    dirs = torch.stack([(i - W * 0.5) / focal, -(j - H * 0.5) / focal, -torch.ones_like(i)], -1)
    rays_d = (dirs[..., None, :] * c2w[:3, :3]).sum(-1)
    rays_o = c2w[:3, -1].expand(rays_d.shape)
    return rays_o, rays_d

def render(model, rays_o, rays_d, near=2.0, far=6.0, N=64):
    t = torch.linspace(near, far, N, device=device)
    z = t.expand(list(rays_d.shape[:-1]) + [N]).clone()
    z = z + torch.rand_like(z) * (far - near) / N           # stratified sampling
    pts = rays_o[..., None, :] + rays_d[..., None, :] * z[..., :, None]
    raw = model(pts)
    rgb, sigma = torch.sigmoid(raw[..., :3]), torch.relu(raw[..., 3])
    dists = torch.cat([z[..., 1:] - z[..., :-1], torch.full_like(z[..., :1], 1e10)], -1)
    alpha = 1.0 - torch.exp(-sigma * dists)
    T = torch.cumprod(torch.cat([torch.ones_like(alpha[..., :1]), 1.0 - alpha + 1e-10], -1), -1)[..., :-1]
    return (alpha * T)[..., None].mul(rgb).sum(-2)

## 3 · Train — watch the PSNR climb
One random view per step: render it, compare to the photo, backprop. PSNR (higher = sharper) is logged.

In [ ]:
model = NeRF().to(device)
opt = torch.optim.Adam(model.parameters(), 5e-4)
psnrs = []; t0 = time.time()
for step in range(STEPS + 1):
    k = np.random.randint(images.shape[0])
    ro, rd = get_rays(H, W, focal, poses[k])
    rgb = render(model, ro, rd)
    loss = ((rgb - images[k]) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % max(1, STEPS // 10) == 0:
        psnr = (-10.0 * torch.log10(loss)).item()
        psnrs.append((step, psnr))
        print(f"step {step:5d}  loss {loss.item():.4f}  PSNR {psnr:5.2f} dB")
print("trained in", round(time.time() - t0, 1), "s")

## 4 · Compare — held-out view vs. ground truth + the PSNR curve

In [ ]:
with torch.no_grad():
    ro, rd = get_rays(H, W, focal, testpose)
    pred = render(model, ro, rd).clamp(0, 1).cpu()
fig, ax = plt.subplots(1, 3, figsize=(11, 3.6))
ax[0].imshow(testimg.cpu()); ax[0].set_title("ground truth"); ax[0].axis("off")
ax[1].imshow(pred);          ax[1].set_title("NeRF render");  ax[1].axis("off")
ax[2].plot(*zip(*psnrs), "-o"); ax[2].set_title("PSNR ↑"); ax[2].set_xlabel("step"); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/B_nerf_from_scratch/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/B_nerf_from_scratch"; os.makedirs(run, exist_ok=True)
torch.save(model.state_dict(), f"{run}/nerf.pt")
json.dump({"psnr": psnrs}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

## (Optional) Publish to the Hugging Face Hub
Upload this run as a **model repo** — the checkpoint, `metrics.json` (full loss/eval history) and the results figure, embedded in an auto-generated model card. Do it for each lab, then group them into a **Collection** on your HF profile (Profile → New collection), or with the commented `add_collection_item` call below. It needs a **write token**, so it only runs once you sign in.

In [ ]:
# (optional) publish this run as a Hugging Face model repo — checkpoint + metrics + plot
!pip -q install huggingface_hub
from huggingface_hub import HfApi, notebook_login
import os
notebook_login()   # paste a WRITE token from https://huggingface.co/settings/tokens
api = HfApi(); user = api.whoami()["name"]
lab = os.path.basename(run); repo_id = f"{user}/" + lab.lower().replace("_", "-")
fig = "\n![results](figure.png)\n" if os.path.exists(f"{run}/figure.png") else ""
open(f"{run}/README.md", "w").write("---\ntags: [ropedia-academy, education]\n---\n# " + lab + "\n\nTrained in **Ropedia Academy** (educational lab). Checkpoint, full loss/eval history (metrics.json) and the results figure are included." + fig)
api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=run, repo_id=repo_id, commit_message="Upload from Ropedia Academy")
print("uploaded ->", "https://huggingface.co/" + repo_id)
# group everything into one Collection (run once, after a few uploads):
# from huggingface_hub import create_collection, add_collection_item
# col = create_collection("Ropedia Academy - trained models", namespace=user, exists_ok=True)
# add_collection_item(col.slug, item_id=repo_id, item_type="model")

## How this links to tracks A–D
A trained NeRF is the **D · Scene / world** scene representation.

### Where to go next
- Add view-dependence (feed ray direction) and a coarse+fine sampler → the original NeRF.
- Swap the MLP for a hash grid → **Instant-NGP** speed.
- For real scenes from your own phone photos, use **[Nerfstudio](https://docs.nerf.studio/)** (`ns-train nerfacto`) or **[3D Gaussian Splatting](https://github.com/graphdeco-inria/gaussian-splatting)**.